In [1]:
import numpy as np
import pandas as pd
import patsy
from tensorzinb.tensorzinb import TensorZINB
import tensorflow as tf

from scipy.stats import norm
from statsmodels.stats.multitest import fdrcorrection


2025-07-02 10:08:32.338441: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 10:08:32.342913: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
fake_cres=pd.read_csv("fake_cres.csv").drop("Unnamed: 0",axis=1)
fake_cres

,CRE,Cell_type,replicate_ID,umi_count
0,nobody,brain,1,0
1,nobody,brain,1,0
2,nobody,brain,1,0
3,nobody,brain,1,0
4,nobody,brain,1,0
...,...,...,...,...
14307,neurogene,blood,3,7
14308,neurogene,blood,3,26
14309,neurogene,blood,3,7
14310,neurogene,blood,3,15


In [3]:
fake_cres_munged=fake_cres
fake_cres_munged["replicate_ID"]=fake_cres_munged["replicate_ID"].map({1:"rep1",2:"rep2",3:"rep3"})

In [4]:
from formulaic import Formula

In [5]:
fake_cres_munged

,CRE,Cell_type,replicate_ID,umi_count
0,nobody,brain,rep1,0
1,nobody,brain,rep1,0
2,nobody,brain,rep1,0
3,nobody,brain,rep1,0
4,nobody,brain,rep1,0
...,...,...,...,...
14307,neurogene,blood,rep3,7
14308,neurogene,blood,rep3,26
14309,neurogene,blood,rep3,7
14310,neurogene,blood,rep3,15


In [6]:
#dense dmatrix approach : works fine for small datasets, but for very large...

#y, X = patsy.dmatrices("umi_count ~ C(CRE)*C(Cell_type)-1",
#                        fake_cres_munged, return_type='dataframe')
#Z = patsy.dmatrix("C(replicate_ID)", fake_cres_munged, return_type='dataframe')

#zinbo=TensorZINB(y["umi_count"].to_numpy().reshape((-1,1)),X,exog_infl=Z.to_numpy())#,same_dispersion=True
#zinb_result=zinbo.fit(init_method="nb")

### simple `formulaic` approach, still using pandas

y, X=Formula("umi_count ~ C(CRE)*C(Cell_type) - 1").get_model_matrix(fake_cres_munged,output='pandas')
Z=Formula('C(replicate_ID)').get_model_matrix(fake_cres_munged,output='pandas')


zinbo=TensorZINB(y["umi_count"].to_numpy().reshape((-1,1)),X.to_numpy(),exog_infl=Z.to_numpy())#,same_dispersion=True
zinb_result=zinbo.fit(init_method="nb")


2025-07-02 10:08:38.285756: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-07-02 10:08:38.285796: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (r814u03n09.mccleary.ycrc.yale.edu): /proc/driver/nvidia/version does not exist
2025-07-02 10:08:38.287222: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-02 10:08:38.339597: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:354] MLIR V1 optimization pass is not enabled


In [7]:
dir(zinbo)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_compute_pi_init',
 '_estimate_dispersion',
 '_nb_init',
 '_no_exog_c',
 '_no_exog_infl',
 '_no_exog_infl_c',
 '_poisson_init',
 '_poisson_init_each',
 'df_model',
 'df_model_each',
 'endog',
 'exog',
 'exog_c',
 'exog_infl',
 'exog_infl_c',
 'fit',
 'k_disperson',
 'k_exog',
 'k_exog_c',
 'k_exog_infl',
 'k_exog_infl_c',
 'loglike_method',
 'nb_only',
 'num_out',
 'num_sample',
 'same_dispersion']

In [8]:
zinb_result

{'llf_total': -21228.78548024269,
 'llfs': array([-21228.78548024]),
 'aic_total': 42485.57096048538,
 'aics': array([42485.57096049]),
 'df_model_total': 14,
 'df': 14,
 'weights': {'x_mu': array([[ 4.6391068 ],
         [ 2.739979  ],
         [ 0.665211  ],
         [ 4.6586823 ],
         [ 2.5245173 ],
         [ 0.03896479],
         [ 1.7861156 ],
         [-0.67995   ],
         [-1.2745893 ],
         [-0.26423696]], dtype=float32),
  'x_pi': array([[ 1.3560897],
         [-1.3262616],
         [ 0.8899107]], dtype=float32),
  'theta': array([[1.2496016]], dtype=float32)},
 'cpu_time': 2.1516470909118652,
 'num_sample': 14312,
 'epochs': 606}

# Recapitulating dispersion constant

In [33]:
np.exp(zinb_result['weights']['theta'][0])

array([3.4889526], dtype=float32)

If I am correct in exponentiating it (I think I am), that's pretty close to the ground truth value of 3.333

# Recapitulating zero-inflation parameters

In [34]:
#I think x_pi are the weights on 
zinb_result['weights']['x_pi']

array([[ 1.3560897],
       [-1.3262616],
       [ 0.8899107]], dtype=float32)

In [35]:
dropout_design_matrix=Z.drop_duplicates().to_numpy()

In [36]:
dropout_design_matrix

array([[1., 0., 0.],
       [1., 1., 0.],
       [1., 0., 1.]])

In [37]:
pis=dropout_design_matrix.dot(zinb_result['weights']['x_pi'])
pis

array([[1.35608971],
       [0.02982807],
       [2.24600041]])

In [38]:
#hrm. I bet these are bernouli constants passed through logit. 
#Let's undo w/ logistic function
1/(1+np.exp(-pis))

array([[0.79512344],
       [0.50745647],
       [0.90430498]])

If we assume the order is rep1, rep2, rep3, then the real values are 0.8, 0.5, 0.9.

That's pretty damn close!

# Recapitulating $\mu$s (mean parameters).

In [9]:
#we begin by getting all combinations of the predictors (which are of course all categorical) present in the data. 
minimal_nb_design = X.drop_duplicates()
minimal_nb_design

,C(CRE)[everybody],C(CRE)[neurogene],C(CRE)[nobody],C(CRE)[redgene],C(CRE)[somebody],C(Cell_type)[T.brain],C(CRE)[T.neurogene]:C(Cell_type)[T.brain],C(CRE)[T.nobody]:C(Cell_type)[T.brain],C(CRE)[T.redgene]:C(Cell_type)[T.brain],C(CRE)[T.somebody]:C(Cell_type)[T.brain]
0,0,0,1,0,0,1,0,1,0,0
440,0,0,0,0,1,1,0,0,0,1
904,1,0,0,0,0,1,0,0,0,0
1355,0,0,0,1,0,1,0,0,1,0
1798,0,1,0,0,0,1,1,0,0,0
2263,0,0,1,0,0,0,0,0,0,0
2762,0,0,0,0,1,0,0,0,0,0
3266,1,0,0,0,0,0,0,0,0,0
3767,0,0,0,1,0,0,0,0,0,0
4262,0,1,0,0,0,0,0,0,0,0


In [40]:
#examining the table above, we reconstruct the index
recapitulated_nb_rate=pd.DataFrame({"cre":["nobody","somebody","everybody","redgene","neurogene","nobody","somebody","everybody","redgene","neurogene"],
    "cell_type":["brain"]*5+["blood"]*5})
recapitulated_nb_rate

,cre,cell_type
0,nobody,brain
1,somebody,brain
2,everybody,brain
3,redgene,brain
4,neurogene,brain
5,nobody,blood
6,somebody,blood
7,everybody,blood
8,redgene,blood
9,neurogene,blood


In [41]:
zinb_result['weights']['x_mu']

array([[ 4.6391068 ],
       [ 2.739979  ],
       [ 0.665211  ],
       [ 4.6586823 ],
       [ 2.5245173 ],
       [ 0.03896479],
       [ 1.7861156 ],
       [-0.67995   ],
       [-1.2745893 ],
       [-0.26423696]], dtype=float32)

In [42]:
len(zinb_result['weights']['x_mu'])

10

That's the same as the number of columns in our design matrix. Assuming orientation was preserved...

In [43]:
np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu']))

,0
0,1.024522
440,9.966656
904,107.562443
1355,30.663034
1798,96.068306
2263,1.944901
2762,12.484867
3266,103.451898
3767,105.496982
4262,15.486660


In [44]:
recapitulated_nb_rate["expected_value"]=np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu'])).to_numpy()
#exponent to undo log link function
recapitulated_nb_rate

,cre,cell_type,expected_value
0,nobody,brain,1.024522
1,somebody,brain,9.966656
2,everybody,brain,107.562443
3,redgene,brain,30.663034
4,neurogene,brain,96.068306
5,nobody,blood,1.944901
6,somebody,blood,12.484867
7,everybody,blood,103.451898
8,redgene,blood,105.496982
9,neurogene,blood,15.486660


Comparing to the ground-truth:

In [45]:
REAL_data = {
    "CRE": ["nobody", "somebody", "everybody", "redgene", "neurogene", "nobody", "somebody", "everybody", "redgene", "neurogene"],
    "Cell-type": ["brain", "brain", "brain", "brain", "brain", "blood", "blood", "blood", "blood", "blood"],
    "mean": [1, 10, 114, 30, 99, 2, 12, 109, 112, 16]
}

# Creating the dataframe
pd.DataFrame(REAL_data)

,CRE,Cell-type,mean
0,nobody,brain,1
1,somebody,brain,10
2,everybody,brain,114
3,redgene,brain,30
4,neurogene,brain,99
5,nobody,blood,2
6,somebody,blood,12
7,everybody,blood,109
8,redgene,blood,112
9,neurogene,blood,16


Pretty good !

# Backcalculating Hessian for SE estimates

In [10]:
def zinb_loglik_tf(params, exog, exog_infl, endog):
    N = endog.shape[0]
    # Extract parameters explicitly
    num_features = exog.shape[1]
    num_infl_features = exog_infl.shape[1]

    x_mu = params[:num_features]
    x_pi = params[num_features:num_features + num_infl_features]
    log_theta = params[-1]
    theta = tf.exp(log_theta)

    mu = tf.exp(tf.matmul(exog, tf.expand_dims(x_mu, axis=-1)))
    pi_logits = tf.matmul(exog_infl, tf.expand_dims(x_pi, axis=-1))
    log_q0 = -tf.nn.softplus(-pi_logits)
    log_q1 = log_q0 - pi_logits

    y = tf.cast(endog, tf.float32)

    # NB log-likelihood
    t1 = tf.math.lgamma(y + theta)
    t2 = -tf.math.lgamma(theta)
    t3 = theta * log_theta
    t4 = y * tf.math.log(mu + 1e-8)
    ty = tf.math.log(mu + theta + 1e-8)
    t5 = -(theta + y) * ty
    nb_case = t1 + t2 + t3 + t4 + t5 + log_q1

    # Zero-inflation likelihood
    p1 = theta * (log_theta - ty) + log_q1
    zero_case = tf.reduce_logsumexp(tf.stack([log_q0, p1], axis=0), axis=0)

    # Combine cases
    ll = tf.where(tf.less(y, 1e-8), zero_case, nb_case)

    # log_likelihood = tf.reduce_sum(ll)

    # 1) get "mean negative log-lik per output" exactly as they do:
    mean_neg_ll = -tf.reduce_mean(ll, axis=0)   # shape = (num_outputs,)

    # 2) recover per-output log-likelihood (add back log-factorial):
    #    note: `endog` is your y_true tensor of shape (N, num_outputs)
    log_fact = tf.reduce_sum(tf.math.lgamma(endog + 1), axis=0)     # ∑ ln(y_i!)
    llfs = -(mean_neg_ll * N + log_fact)                            # shape = (num_outputs,)

    # 3) sum across outputs:
    log_likelihood = tf.reduce_sum(llfs)
    return log_likelihood

In [11]:
x_mu = zinb_result['weights']['x_mu'].flatten()
theta = zinb_result['weights']['theta'].flatten()
x_pi = zinb_result['weights']['x_pi'].flatten()
params = np.concatenate([x_mu, x_pi, theta])

In [16]:
params

array([ 4.6391068 ,  2.739979  ,  0.665211  ,  4.6586823 ,  2.5245173 ,
        0.03896479,  1.7861156 , -0.67995   , -1.2745893 , -0.26423696,
        1.3560897 , -1.3262616 ,  0.8899107 ,  1.2496016 ], dtype=float32)

In [13]:
endog = y["umi_count"].to_numpy().reshape((-1,1))
exog = X.to_numpy()
exog_infl = Z.to_numpy()

In [14]:
exog_tensor = tf.constant(exog, dtype=tf.float32)
endog_tensor = tf.constant(endog, dtype=tf.float32)
exog_infl_tensor = tf.constant(exog_infl, dtype=tf.float32)
params_tensor = tf.Variable(params, dtype=tf.float32)

In [15]:
params_tensor

<tf.Variable 'Variable:0' shape=(14,) dtype=float32>

- tf.GradientTape() is TensorFlow’s automatic-differentiation context:
    - The inner tape (tape1) records all operations needed to compute ll (the scalar log-likelihood) with respect to our parameter vector params_tensor.
    - After computing ll, tape1.gradient(ll, params_tensor) gives us the gradient ∂LL/∂params (a vector of first derivatives).
    - The outer tape (tape2) then records the computation of that gradient with respect to params_tensor again, enabling us to compute the Jacobian of the gradient, which is exactly the Hessian matrix (matrix of second derivatives ∂²LL/∂params²).
- Why? For a Wald test we need the Fisher information or Hessian.  Approximating the Hessian numerically via automatic differentiation is the most direct way to get the curvature of the log-likelihood at the fitted parameters.

//


- Because the tensorZINB code called disable_eager_execution(), our computation graph is in graph mode, not in eager mode.
- In graph mode, tensors like hessian are just symbolic objects; you can’t call .numpy() on them.
- Opening a TF1 session (tf.compat.v1.Session()) gives us an environment in which we can actually evaluate those symbolic tensors.
- global_variables_initializer() ensures any TensorFlow variables (including params_tensor) are initialized in that session.

In [17]:
import tensorflow as tf
import numpy as np

# (Assuming you’ve already defined zinb_loglik_tf, built exog_tensor, exog_infl_tensor, endog_tensor,
#  and have params_tensor set up exactly as before.)

# 1) Build the Hessian in graph mode
with tf.GradientTape() as tape2:
    with tf.GradientTape() as tape1:
        ll = zinb_loglik_tf(params_tensor, exog_tensor, exog_infl_tensor, endog_tensor)
    grad = tape1.gradient(ll, params_tensor)
hessian = tape2.jacobian(grad, params_tensor)  # this is now a Tensor in the graph

# 2) Create and initialize a TF1 Session
sess = tf.compat.v1.Session()
sess.run(tf.compat.v1.global_variables_initializer())

# 3) Evaluate the Hessian tensor
hessian_val = sess.run(hessian)

# 4) Compute covariance and standard errors
cov_matrix = np.linalg.inv(-hessian_val)
standard_errors = np.sqrt(np.diag(cov_matrix))

print("Standard errors:\n", standard_errors)

Standard errors:
 [0.02769498 0.03023117 0.05174283 0.02709585 0.03044432 0.0398754
 0.05739546 0.09363861 0.05644304 0.06058175 0.0374013  0.04809219
 0.06305671 0.02848125]


In [71]:
ll

<tf.Tensor 'Sum_7:0' shape=() dtype=float32>

In [72]:
sess.run(ll)

-21228.75

In [44]:
zinb_result['llf_total']

-21228.78548024269

In [19]:
standard_errors.shape

(14,)

In [20]:
params.shape

(14,)

# Wald test

In [22]:
beta = x_mu
se_beta = standard_errors[:beta.shape[0]]

In [73]:
se_beta

array([0.02769498, 0.03023117, 0.05174283, 0.02709585, 0.03044432,
       0.0398754 , 0.05739546, 0.09363861, 0.05644304, 0.06058175],
      dtype=float32)

In [29]:

# 1) Wald Z‐statistics
z_scores = beta / se_beta

# 2) Two‐sided p‐values
p_values = 2 * (1 - norm.cdf(np.abs(z_scores)))

# 3) FDR correction (Benjamini‐Hochberg)
rejected, p_adjusted = fdrcorrection(p_values, alpha=0.05)

# 4) Summarize results in a structured way
import pandas as pd

results = pd.DataFrame({
    'beta':            beta,
    'SE':              se_beta,
    'Z':               z_scores,
    'p_raw':           p_values,
    'p_adj (FDR=0.05)': p_adjusted,
    'significant':     rejected
})
results.index.name = 'parameter_index'
print(results)

                     beta        SE           Z         p_raw  \
parameter_index                                                 
0                4.639107  0.027695  167.507126  0.000000e+00   
1                2.739979  0.030231   90.634232  0.000000e+00   
2                0.665211  0.051743   12.856100  0.000000e+00   
3                4.658682  0.027096  171.933426  0.000000e+00   
4                2.524517  0.030444   82.922432  0.000000e+00   
5                0.038965  0.039875    0.977164  3.284882e-01   
6                1.786116  0.057395   31.119457  0.000000e+00   
7               -0.679950  0.093639   -7.261428  3.830269e-13   
8               -1.274589  0.056443  -22.581871  0.000000e+00   
9               -0.264237  0.060582   -4.361660  1.290796e-05   

                 p_adj (FDR=0.05)  significant  
parameter_index                                 
0                    0.000000e+00         True  
1                    0.000000e+00         True  
2                    0.

In [87]:
df=pd.DataFrame({"cre":["nobody","somebody","everybody","redgene","neurogene","nobody","somebody","everybody","redgene","neurogene"],
    "cell_type":["brain"]*5+["blood"]*5})

In [91]:
pd.concat([df, results], axis=1)


,cre,cell_type,beta,SE,Z,p_raw,p_adj (FDR=0.05),significant
0,nobody,brain,4.639107,0.027695,167.507126,0.000000e+00,0.000000e+00,True
1,somebody,brain,2.739979,0.030231,90.634232,0.000000e+00,0.000000e+00,True
2,everybody,brain,0.665211,0.051743,12.856100,0.000000e+00,0.000000e+00,True
3,redgene,brain,4.658682,0.027096,171.933426,0.000000e+00,0.000000e+00,True
4,neurogene,brain,2.524517,0.030444,82.922432,0.000000e+00,0.000000e+00,True
5,nobody,blood,0.038965,0.039875,0.977164,3.284882e-01,3.284882e-01,False
6,somebody,blood,1.786116,0.057395,31.119457,0.000000e+00,0.000000e+00,True
7,everybody,blood,-0.679950,0.093639,-7.261428,3.830269e-13,4.787837e-13,True
8,redgene,blood,-1.274589,0.056443,-22.581871,0.000000e+00,0.000000e+00,True
9,neurogene,blood,-0.264237,0.060582,-4.361660,1.290796e-05,1.434218e-05,True
